# 00 · Setup and configuration
**Random splits flatter deep sequence models: a per-class chronological benchmark for intrusion detection in software-defined networks**

Run this notebook once. It installs the libraries, records your hardware (for the *Experimental environment* section), and writes `config.json`, which every other notebook reads.

**Run order:** 00 → 01 → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09 → 10. The paper used `"mode": "reduced"`. Every experiment notebook is *resumable*: if it stops, run it again and it skips finished runs.

## 1. Install the libraries (first time only)

In [ ]:
%pip install -q -r ../requirements.txt

## 2. Point to your datasets
Put each dataset in its own folder (sub-folders are fine; every `.csv` inside is read).

| Dataset | Files expected | Notes |
|---|---|---|
| **InSDN** (primary) | `Normal_data.csv`, `OVS.csv`, `metasploitable-2.csv` | 8 classes |
| **CICIDS2017** | the corrected CSVs from Engelen et al. (2021) | the original release has labelling and timestamp faults; use the corrected one for chronological splits |
| **CSE-CIC-IDS2018** | daily CSVs (corrected release preferred) | optional; ~16 M flows, so it is sampled per class (`max_rows_per_class`) |

Leave a `path` empty to skip that dataset.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))
from sdnids_bench.config import DEFAULT_CONFIG, deep_update, save_config, capture_environment

cfg = deep_update(DEFAULT_CONFIG, {
    "datasets": {
        "InSDN":           {"path": r"C:/path/to/InSDN"},            # <-- EDIT
        "CICIDS2017":      {"path": r"C:/path/to/CICIDS2017_improved"},  # <-- EDIT (corrected release)
        "CSE-CIC-IDS2018": {"path": r""},                            # <-- optional, not used in the paper
    },
    "out_dir": "results",
    # "full"    = design specification (10 seeds, 40 tuning trials, L up to 32) -> publishable, needs a GPU or ~2 weeks of CPU
    # "reduced" = publishable on a CPU laptop: 10 seeds, 20 trials, L up to 16, random-split arms 5 seeds,
    #             CICIDS2017 training capped at 150,000 flows (state these choices in the methods section)
    # "quick"   = a short pass (3 seeds, 8 trials, 60k training flows) to check everything runs - not for the paper
    # Use a different out_dir for each mode, e.g. "results_quick" / "results"
    "mode": "reduced",
    "device": "auto",     # uses an NVIDIA GPU (cuda) or Apple GPU (mps) when available
})
for name, d in cfg["datasets"].items():
    ok = bool(d["path"]) and os.path.isdir(d["path"])
    n = sum(f.lower().endswith('.csv') for _, _, fs in os.walk(d["path"]) for f in fs) if ok else 0
    print(f"{name:16s} {'FOUND' if ok else 'skipped':8s} {n} CSV files  {d['path']}")
print('config written to', save_config(cfg, '..'))

## 3. Record the computing environment

In [ ]:
env = capture_environment(os.path.join('..', cfg['out_dir']))
for k, v in env.items():
    print(f'{k:22s} {v}')

## 4. What will run (full mode; the paper used reduced mode, see the README)

| Notebook | Experiment | Runs (approx.) |
|---|---|---|
| 02 | Equal-budget tuning: 40 Optuna trials × 9 models × 2 tasks × each dataset | 720 per dataset |
| 03 | **B3** window length L ∈ {1, 4, 8, 16, 32}, 5 seeds, InSDN | 225 |
| 04 | **B1/B2** 3 protocols × 2 tasks × L ∈ {1, best L} × 10 seeds, every dataset (S-C arm = main benchmark; other datasets = **B6** replication) | 1,080 per dataset |
| 05 | **B4** 6 imbalance strategies × top-4 models × 5 seeds | ~110 |
| 06 | **B5** computational cost, every model | 27 |
| 07 | **S1** sensitivity arms: S-C on raw data (both datasets), S-C after label-free deduplication (CICIDS2017), 10 seeds | 270 |
| 08 | Data-preparation audit (no training) | — |
| 09 | Figures (PDF + PNG) and descriptive tables (no training) | — |
| 10 | Every statistic and interval reported in the paper (no training) | — |

Tip: on a single GPU, InSDN takes roughly one to two days end to end; CICIDS2017 several more. Close other heavy programs while notebook 06 measures latency.